In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from infer_ci import MetricEvaluator, classification_report_with_ci

# 1. 샘플 데이터셋 준비 (이진 분류: 유방암 진단 데이터)
# 신뢰구간(CI) 평가를 진행하기에 적절한 샘플 크기의 데이터셋입니다.
data = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.25, random_state=42
)

print(f"Train samples: {len(X_train)}, Test samples: {len(X_test)}")

Train samples: 426, Test samples: 143


In [2]:
# 2. Baseline 모델 (Random Forest) 학습
model = RandomForestClassifier(n_estimators=50, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("Model training & prediction complete.")

Model training & prediction complete.


In [3]:
# 3. infer-ci MetricEvaluator를 이용한 평가
evaluator = MetricEvaluator()

# 평가 지표 선정 이유:
# - Accuracy: 전체 예측 정확도
# - Precision & Recall: 양성 클래스(진단)에 대한 정밀도 및 재현율
# - F1 Score: Precision과 Recall의 균형 평가
metrics = ['accuracy', 'precision', 'recall', 'f1']

print("=== infer-ci Evaluation Results (95% Bootstrap BCa CI) ===")
for metric in metrics:
    score, ci = evaluator.evaluate(
        y_true=y_test,
        y_pred=y_pred,
        task='classification',
        metric=metric,
        method='bootstrap_bca'
    )
    print(f"{metric.upper():<10} | Point Estimate: {score:.4f} | 95% CI: [{ci[0]:.4f}, {ci[1]:.4f}]")

=== infer-ci Evaluation Results (95% Bootstrap BCa CI) ===


Bootstrap CI (BCA): 100%|████████████████████████████████████████████████████████████████████████| 10143/10143 [00:00<00:00]


ACCURACY   | Point Estimate: 0.9720 | 95% CI: [0.9301, 0.9930]


Bootstrap CI (BCA): 100%|████████████████████████████████████████████████████████████████████████| 10143/10143 [00:02<00:00]


PRECISION  | Point Estimate: 0.9885 | 95% CI: [0.9390, 1.0000]


Bootstrap CI (BCA): 100%|████████████████████████████████████████████████████████████████████████| 10143/10143 [00:02<00:00]


RECALL     | Point Estimate: 0.9663 | 95% CI: [0.9070, 0.9896]


Bootstrap CI (BCA): 100%|████████████████████████████████████████████████████████████████████████| 10143/10143 [00:02<00:00]

F1         | Point Estimate: 0.9720 | 95% CI: [0.9301, 0.9930]


In [4]:
# 4. classification_report_with_ci를 이용한 종합 리포트 출력
report = classification_report_with_ci(y_test, y_pred)
print("\n=== Comprehensive Classification Report with CI ===")
print(report)


=== Comprehensive Classification Report with CI ===
     Class  Precision  Recall  F1-Score    Precision CI       Recall CI  \
0  Class 0      0.946   0.981     0.964  (0.854, 0.982)  (0.902, 0.997)   
1  Class 1      0.989   0.966     0.977  (0.938, 0.998)  (0.906, 0.988)   
2    micro      0.972   0.972     0.972  (0.945, 0.999)  (0.945, 0.999)   
3    macro      0.967   0.974     0.970  (0.932, 1.003)  (0.942, 1.006)   

      F1-Score CI  Support  
0  (0.918, 1.009)       54  
1  (0.945, 1.009)       89  
2  (0.945, 0.999)      143  
3  (0.852, 1.089)      143  
